# Govern a Colab regression model with ML Sentry

This notebook trains a regression model, reports its evaluation to **your ML Sentry
server**, and acts as the **external training runner**: when a human approves
retraining in the ML Sentry UI, this notebook claims the approved job, retrains,
evaluates the candidate on the champion's fixed holdout and reports it back.

Project: https://github.com/rishsoraganvi/ML_Sentry

**Before you start** (details:
[docs/colab.md](https://github.com/rishsoraganvi/ML_Sentry/blob/main/docs/colab.md)):
1. ML Sentry runs on your machine (`docker compose up -d`) with a policy for
   `colab_regressor` (`configs/my-model.example.yaml`) and
   `ML_SENTRY_RETRAINING_ENABLED=true` (needed for the external runner).
2. Colab can reach it over HTTPS, e.g. `cloudflared tunnel --url http://127.0.0.1:8000`.
3. In Colab's 🔑 *Secrets* panel add `ML_SENTRY_URL`, `ML_SENTRY_TOKEN` (service
   token) and `ML_SENTRY_RUNNER_TOKEN`. **Never put a reviewer token in Colab.**

In [ ]:
%%writefile ml_sentry_client.py
"""Single-file ML Sentry client (Python standard library only).

Copy this file next to your training/evaluation code (or into Google Colab) to:

* submit regression evaluations          -> MLSentryClient(url, token).submit_evaluation(...)
* act as an external training runner     -> MLSentryClient(url, runner_token=...).run_jobs(...)

It never needs a reviewer token: approvals are made by humans in the ML Sentry UI.
"""

from __future__ import annotations

import json
import math
import time
import urllib.error
import urllib.request
from collections.abc import Callable, Sequence
from datetime import UTC, datetime
from typing import Any

__all__ = ["MLSentryClient", "MLSentryError", "regression_metrics"]


class MLSentryError(Exception):
    def __init__(self, status: int, code: str, message: str) -> None:
        super().__init__(f"HTTP {status} {code}: {message}")
        self.status, self.code, self.message = status, code, message


def regression_metrics(y_true: Sequence[float], y_pred: Sequence[float]) -> dict[str, float]:
    """MAE, MSE, RMSE and R² (R² omitted for a constant target), as ML Sentry defines them."""
    if len(y_true) != len(y_pred) or not y_true:
        raise ValueError("y_true and y_pred must be non-empty and the same length")
    n = len(y_true)
    errors = [float(t) - float(p) for t, p in zip(y_true, y_pred, strict=True)]
    mse = math.fsum(e * e for e in errors) / n
    metrics = {"mae": math.fsum(abs(e) for e in errors) / n, "mse": mse, "rmse": math.sqrt(mse)}
    mean = math.fsum(float(t) for t in y_true) / n
    ss_tot = math.fsum((float(t) - mean) ** 2 for t in y_true)
    if ss_tot > 0:
        metrics["r2"] = 1 - (mse * n) / ss_tot
    return metrics


class MLSentryClient:
    def __init__(
        self,
        base_url: str,
        token: str | None = None,
        *,
        runner_token: str | None = None,
        timeout: float = 30.0,
        extra_headers: dict[str, str] | None = None,
    ) -> None:
        self.base_url = base_url.rstrip("/")
        self.token = token
        self.runner_token = runner_token
        self.timeout = timeout
        self.extra_headers = extra_headers or {}

    # -- HTTP -----------------------------------------------------------------------

    def _call(self, method: str, path: str, body: Any = None, *, runner: bool = False) -> Any:
        token = self.runner_token if runner else self.token
        if not token:
            raise ValueError("this call needs " + ("runner_token" if runner else "token"))
        data = None if body is None else json.dumps(body, allow_nan=True).encode()
        request = urllib.request.Request(  # noqa: S310 - the operator supplies the URL
            f"{self.base_url}{path}",
            data=data,
            method=method,
            headers={
                "Authorization": f"Bearer {token}",
                "Content-Type": "application/json",
                **self.extra_headers,
            },
        )
        try:
            with urllib.request.urlopen(request, timeout=self.timeout) as response:  # noqa: S310
                raw = response.read()
        except urllib.error.HTTPError as err:
            try:
                error = json.loads(err.read()).get("error", {})
            except ValueError:
                error = {}
            raise MLSentryError(
                err.code, error.get("code", "HTTP_ERROR"), error.get("message", str(err))
            ) from None
        return json.loads(raw) if raw else None

    # -- evaluations ----------------------------------------------------------------

    def submit_evaluation(
        self,
        *,
        event_id: str,
        model_name: str,
        model_version: str,
        target_name: str,
        dataset_id: str,
        dataset_version: str,
        evaluation_protocol_id: str,
        sample_count: int,
        metrics: dict[str, float],
        role: str = "monitoring",
        evaluation_timestamp: datetime | None = None,
        **optional: Any,
    ) -> dict[str, Any]:
        """Send one evaluation. Re-sending the same event_id with the same data is safe."""
        timestamp = evaluation_timestamp or datetime.now(UTC)
        event = {
            "schema_version": 1,
            "event_id": event_id,
            "role": role,
            "model_name": model_name,
            "model_version": model_version,
            "task_type": "regression",
            "target_name": target_name,
            "dataset_id": dataset_id,
            "dataset_version": dataset_version,
            "evaluation_protocol_id": evaluation_protocol_id,
            "evaluation_timestamp": timestamp.isoformat(),
            "sample_count": int(sample_count),
            "metrics": {k: float(v) for k, v in metrics.items()},
            **optional,
        }
        result: dict[str, Any] = self._call("POST", "/api/v1/evaluations/events", event)
        return result

    def submit_mlflow_run(self, run_id: str, role: str | None = None) -> dict[str, Any]:
        result: dict[str, Any] = self._call(
            "POST", "/api/v1/evaluations/mlflow", {"run_id": run_id, "role": role}
        )
        return result

    def incident(self, incident_id: str) -> dict[str, Any]:
        result: dict[str, Any] = self._call("GET", f"/api/v1/incidents/{incident_id}")
        return result

    # -- external runner ------------------------------------------------------------

    def queued_jobs(self, model_name: str | None = None) -> list[dict[str, Any]]:
        query = f"?runner=external&model_name={model_name}" if model_name else "?runner=external"
        result: list[dict[str, Any]] = self._call("GET", f"/api/v1/jobs{query}", runner=True)
        return result

    def claim(self, job_id: str) -> dict[str, Any]:
        result: dict[str, Any] = self._call("POST", f"/api/v1/jobs/{job_id}/claim", runner=True)
        return result

    def report(self, job_id: str, result: dict[str, Any]) -> dict[str, Any]:
        response: dict[str, Any] = self._call(
            "POST", f"/api/v1/jobs/{job_id}/result", result, runner=True
        )
        return response

    def run_jobs(
        self,
        train_and_evaluate: Callable[[dict[str, Any]], dict[str, Any]],
        *,
        model_name: str,
        wait_seconds: float = 0,
        poll_seconds: float = 10,
        max_jobs: int = 1,
        log: Callable[[str], None] = print,
    ) -> list[dict[str, Any]]:
        """Claim approved jobs for ``model_name`` and run ``train_and_evaluate(spec)``.

        ``train_and_evaluate`` receives the approved job spec (pipeline config, training
        dataset, and the champion's holdout identity) and must return::

            {"candidate_model_ref": "...", "candidate_model_version": "...",
             "metrics": {...}, "sample_count": N, "candidate_run_id": "..." (optional)}

        with the metrics computed on that SAME holdout. Exceptions are reported as a
        failed job (error code only; no traceback leaves this process).
        """
        deadline = time.monotonic() + wait_seconds
        finished: list[dict[str, Any]] = []
        while len(finished) < max_jobs:
            jobs = self.queued_jobs(model_name)
            if not jobs:
                if time.monotonic() >= deadline:
                    break
                time.sleep(poll_seconds)
                continue
            job = self.claim(jobs[0]["id"])
            spec, holdout = job["spec"], job["spec"]["holdout"]
            log(f"Claimed job {job['id']} ({spec['pipeline_id']}); training...")
            try:
                out = train_and_evaluate(spec)
                result = {
                    "status": "succeeded",
                    "candidate_model_ref": out["candidate_model_ref"],
                    "candidate_model_version": out["candidate_model_version"],
                    "candidate_run_id": out.get("candidate_run_id"),
                    "evaluation": {
                        "dataset_id": holdout["dataset_id"],
                        "dataset_version": holdout["dataset_version"],
                        "evaluation_protocol_id": holdout["evaluation_protocol_id"],
                        "target_name": holdout["target_name"],
                        "sample_count": int(out["sample_count"]),
                        "metrics": {k: float(v) for k, v in out["metrics"].items()},
                    },
                }
            except Exception as exc:  # report, don't leak details
                log(f"Training failed: {type(exc).__name__}")
                result = {"status": "failed", "error_code": type(exc).__name__.upper()[:64]}
            finished.append(self.report(job["id"], result))
            log(f"Reported job {job['id']}: {finished[-1]['status']}")
        return finished


In [ ]:
# Configuration. In Colab the values come from the Secrets panel; elsewhere from env vars.
import os
import time
import uuid

try:
    from google.colab import userdata  # type: ignore[import-not-found]

    def secret(name: str) -> str:
        return str(userdata.get(name))
except ImportError:  # running outside Colab (e.g. the local test of this notebook)

    def secret(name: str) -> str:
        return os.environ[name]


from ml_sentry_client import MLSentryClient, regression_metrics

sentry = MLSentryClient(
    secret("ML_SENTRY_URL"),
    secret("ML_SENTRY_TOKEN"),
    runner_token=secret("ML_SENTRY_RUNNER_TOKEN"),
)
MODEL_NAME = os.environ.get("ML_SENTRY_MODEL", "colab_regressor")
RUN = os.environ.get("ML_SENTRY_RUN_ID", uuid.uuid4().hex[:6])  # keeps re-runs distinct
SEED = 42

## 1. Data and a fixed, versioned holdout
Candidates are only ever compared with the champion on **this exact holdout**, so
its identity (dataset id + version + protocol + target) is part of every report.

In [ ]:
from sklearn.datasets import fetch_california_housing, make_regression
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split

if os.environ.get("ML_SENTRY_SYNTHETIC_DATA"):  # offline fallback used by the local test
    X, y = make_regression(n_samples=4000, n_features=8, noise=20.0, random_state=SEED)
    y = (y - y.min()) / 100.0
    DATASET_ID = "synthetic-regression"
else:
    X, y = fetch_california_housing(return_X_y=True)
    DATASET_ID = "california-housing"

X_train, X_hold, y_train, y_hold = train_test_split(X, y, test_size=0.2, random_state=SEED)
HOLDOUT = {
    "dataset_id": DATASET_ID,
    "dataset_version": f"v1-seed{SEED}",
    "evaluation_protocol_id": "holdout-20pct-v1",
    "target_name": "median_house_value",
}
print(f"train={len(y_train)} holdout={len(y_hold)}")


def evaluate(model) -> dict[str, float]:
    return regression_metrics(list(y_hold), list(model.predict(X_hold)))

## 2. Train the champion and register its baseline evaluation

In [ ]:
champion = HistGradientBoostingRegressor(max_iter=100, random_state=SEED).fit(X_train, y_train)
champion_metrics = evaluate(champion)
print("champion", {k: round(v, 4) for k, v in champion_metrics.items()})

decision = sentry.submit_evaluation(
    event_id=f"{MODEL_NAME}-{RUN}-baseline",
    role="baseline",
    model_name=MODEL_NAME,
    model_version=f"1-{RUN}",
    sample_count=len(y_hold),
    metrics=champion_metrics,
    **HOLDOUT,
)
print("baseline:", decision["severity"], decision["proposed_action"])

## 3. Report a monitoring evaluation
In production you would evaluate the deployed model on new labelled data here.
To demonstrate a regression we evaluate a deliberately weak model version.

In [ ]:
degraded = HistGradientBoostingRegressor(max_iter=5, random_state=SEED).fit(X_train, y_train)
monitoring = sentry.submit_evaluation(
    event_id=f"{MODEL_NAME}-{RUN}-monitoring",
    model_name=MODEL_NAME,
    model_version=f"2-{RUN}",
    sample_count=len(y_hold),
    metrics=evaluate(degraded),
    **HOLDOUT,
)
print(monitoring["severity"], "->", monitoring["proposed_action"])
print(monitoring["decision_reason"])
INCIDENT_ID = monitoring["incident_id"]

## 4. A human approves retraining in the ML Sentry UI
Open `<your ML Sentry URL>/ui/` on your machine, enter **your reviewer token**,
review the retraining request (evidence, pipeline, holdout) and approve it.
Nothing in this notebook can approve anything.

## 5. Run the approved job here (external runner)
The next cell waits for the approved job, trains with the **approved** config and
reports the candidate's metrics on the same holdout.

In [ ]:
def train_and_evaluate(spec: dict) -> dict:
    assert {k: spec["holdout"][k] for k in HOLDOUT} == HOLDOUT, "job is for another holdout"
    config = spec["config"]  # approved, allow-listed configuration from the policy
    model = HistGradientBoostingRegressor(
        max_iter=int(config.get("max_iter", 300)),
        learning_rate=float(config.get("learning_rate", 0.1)),
        random_state=SEED,
    ).fit(X_train, y_train)
    version = f"3-{RUN}"
    # Save the model wherever you keep artifacts (e.g. Google Drive) and reference it here.
    return {
        "candidate_model_ref": f"colab://{MODEL_NAME}/{version}",
        "candidate_model_version": version,
        "metrics": evaluate(model),
        "sample_count": len(y_hold),
    }


results = sentry.run_jobs(
    train_and_evaluate,
    model_name=MODEL_NAME,
    wait_seconds=float(os.environ.get("ML_SENTRY_WAIT_SECONDS", 600)),
    poll_seconds=5,
)
print([r["status"] for r in results])

## 6. A human approves the deployment
If the candidate passed every gate (same holdout, no regression vs the champion,
primary metric improved), ML Sentry asks for a **second, separate** approval in
the UI. After that it releases to the configured target and verifies it.

In [ ]:
for _ in range(int(os.environ.get("ML_SENTRY_STATUS_POLLS", 1))):
    status = sentry.incident(INCIDENT_ID)
    print(status["status"], "/", status["workflow_status"])
    if status["status"] == "resolved":
        break
    time.sleep(5)